# Active Learning: entropy vs random

Классификация темы новости по колонке `category` на подмножестве `unified_news.csv` (классы с частотой ≥ 10). Старт N=50, 5 итераций по 20 примеров, фиксированный test 20%.

In [ ]:
import os
import sys

_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, "agents")):
    ROOT = _cwd
else:
    ROOT = os.path.abspath(os.path.join(_cwd, ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
import matplotlib.pyplot as plt

from agents.al_agent import ActiveLearningAgent, prepare_al_data

RANDOM_STATE = 42
CSV_PATH = os.path.join(ROOT, "data", "raw", "unified_news.csv")

df = pd.read_csv(CSV_PATH)
labeled_df, pool_df, test_df = prepare_al_data(
    df,
    text_col="text",
    label_col="category",
    min_class_count=10,
    test_size=0.2,
    initial_labeled=50,
    random_state=RANDOM_STATE,
)
len(labeled_df), len(pool_df), len(test_df)

In [ ]:
agent_ent = ActiveLearningAgent(model="logreg", random_state=RANDOM_STATE)
hist_entropy = agent_ent.run_cycle(
    labeled_df,
    pool_df,
    test_df,
    strategy="entropy",
    n_iterations=5,
    batch_size=20,
)
pd.DataFrame(hist_entropy)

In [ ]:
agent_rand = ActiveLearningAgent(model="logreg", random_state=RANDOM_STATE)
hist_random = agent_rand.run_cycle(
    labeled_df,
    pool_df,
    test_df,
    strategy="random",
    n_iterations=5,
    batch_size=20,
)
pd.DataFrame(hist_random)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
xe = [h["n_labeled"] for h in hist_entropy]
xr = [h["n_labeled"] for h in hist_random]
axes[0].plot(xe, [h["accuracy"] for h in hist_entropy], "o-", label="entropy")
axes[0].plot(xr, [h["accuracy"] for h in hist_random], "s--", label="random")
axes[0].set_xlabel("n_labeled")
axes[0].set_ylabel("accuracy")
axes[0].set_title("Accuracy")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(xe, [h["f1"] for h in hist_entropy], "o-", label="entropy")
axes[1].plot(xr, [h["f1"] for h in hist_random], "s--", label="random")
axes[1].set_xlabel("n_labeled")
axes[1].set_ylabel("F1 macro")
axes[1].set_title("F1 macro")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
out_png = os.path.join(ROOT, "data", "processed", "al_entropy_vs_random.png")
os.makedirs(os.path.dirname(out_png), exist_ok=True)
fig.savefig(out_png, dpi=150)
plt.show()
print("Сохранено:", out_png)

In [ ]:
def savings_at_metric(hist_a, hist_b, key="accuracy"):
    """Сколько примеров экономит стратегия A относительно B при том же качестве.
    Берём финальное качество B на последней точке и ищем минимальный n_labeled у A с качеством >= этого порога.
    """
    target = hist_b[-1][key]
    n_b = hist_b[-1]["n_labeled"]
    for h in hist_a:
        if h[key] >= target:
            return {
                "metric": key,
                "target_from_random_final": target,
                "n_random_final": n_b,
                "n_entropy_to_reach": h["n_labeled"],
                "saved_labels": n_b - h["n_labeled"],
            }
    return {
        "metric": key,
        "target_from_random_final": target,
        "note": "entropy не достиг порога random на наблюдаемых точках",
    }

print("Accuracy:", savings_at_metric(hist_entropy, hist_random, "accuracy"))
print("F1:", savings_at_metric(hist_entropy, hist_random, "f1"))

### Бонус: LLM-комментарий к выборке (если задан `ANTHROPIC_API_KEY`)

Ниже — один раз переобучим на стартовых 50, возьмём первые 3 индекса из entropy-query и запросим пояснение.

In [ ]:
import os

probe = ActiveLearningAgent(model="logreg", random_state=RANDOM_STATE)
probe.fit(labeled_df)
idx = probe.query(pool_df, "entropy", batch_size=3)
if os.environ.get("ANTHROPIC_API_KEY"):
    print(probe.explain_selection(pool_df, idx, "entropy"))
else:
    print("Пропуск: задайте ANTHROPIC_API_KEY для вызова Claude.")